# Wine Quality Prediction: A Comparative Analysis of Machine Learning Models

## 1. Introduction and Business Context
The objective of this project is to analyze the **Red Wine Quality Dataset** (Cortez et al., 2009), https://www.kaggle.com/datasets/uciml/red-wine-quality-cortez-et-al-2009. The dataset contains 1,599 samples of red wine, each evaluated on 11 physicochemical properties (such as alcohol content, acidity, and pH) and assigned a sensory quality score ranging from 3 to 8.

**The Strategic Approach:**
While the original dataset presents a regression problem, predicting exact sensory scores can be highly subjective. To simulate a more practical business use-case—such as a winery automatically identifying top-tier wines for premium pricing before bottling—we will transform this into a **Binary Classification** task:
* **Premium (Class 1):** Wines with a quality score of 7 or higher.
* **Standard (Class 0):** Wines with a quality score of 6 or lower.

We will conduct a rigorous Exploratory Data Analysis (EDA) followed by the implementation and cross-validation of multiple machine learning models (KNN, SVM, Decision Trees, Random Forests, and a Neural Network) to determine the best predictive algorithm.

### 1.1 Dataset Overview

| Property | Value |
| :--- | :--- |
| **Samples** | 1,599 |
| **Features** | 11 physicochemical properties |
| **Target** | Binary: Premium (quality ≥ 7) vs Standard (quality < 7) |
| **Class Balance** | ~13.6% Premium, ~86.4% Standard (heavily imbalanced) |
| **Source** | UCI Machine Learning Repository (Cortez et al., 2009) |

In [ ]:
# ==============================================================================
# LIBRARY IMPORTS (Consolidated)
# ==============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn: Preprocessing & Model Selection
from sklearn.model_selection import train_test_split, GridSearchCV, validation_curve
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier

# Scikit-learn: Models
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier

# Scikit-learn: Metrics
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score,
    roc_curve, auc, precision_recall_curve, average_precision_score
)

# Imbalanced-learn: SMOTE for class imbalance
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# Plotting configuration
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("All libraries successfully imported!")

In [ ]:
# Load the dataset
df = pd.read_csv('winequality-red.csv')

print(f"Original dataset dimensions: {df.shape}")

# Transform the Target into a Binary Classification problem
df['is_premium'] = (df['quality'] >= 7).astype(int)
df = df.drop('quality', axis=1)

display(df.head())

## 2. Exploratory Data Analysis (EDA)

### 2.1 Data Quality Checks
Before any modeling, we verify data integrity: missing values, duplicates, and basic statistics. Even when the dataset is known to be clean, demonstrating this step is essential for reproducibility and rigor.

In [ ]:
# Missing values
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nTotal missing values: {df.isnull().sum().sum()}")

# Duplicates
n_dupes = df.duplicated().sum()
print(f"\nDuplicate rows: {n_dupes}")
if n_dupes > 0:
    print(f"  -> Removing {n_dupes} duplicate rows.")
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"  -> New shape: {df.shape}")

# Basic statistics
display(df.describe().round(3))

### 2.2 Target Variable Distribution
A severe class imbalance can heavily skew model performance, causing algorithms to favor the majority class. We visualize the distribution to motivate our later choice of SMOTE resampling and F1-Score optimization.

In [ ]:
class_counts = df['is_premium'].value_counts()
print("Class Distribution:\n", class_counts)
print(f"\nPercentage of Premium wines: {(class_counts[1] / len(df)) * 100:.2f}%")
print(f"Imbalance Ratio: {class_counts[0] / class_counts[1]:.1f} : 1\n")

plt.figure(figsize=(6, 4))
sns.countplot(x='is_premium', data=df, palette='viridis')
plt.title('Distribution of Wine Classes: Standard (0) vs Premium (1)')
plt.xlabel('Wine Class')
plt.ylabel('Number of Samples')
plt.show()

### 2.3 Feature Distributions and Outlier Analysis
Understanding the distribution and scale of each feature is critical. We look for skewness, outliers, and the scale differences that motivate standardization for distance-based models.

In [ ]:
# Distribution plots for all features
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()

features = df.columns.drop('is_premium')
for i, col in enumerate(features):
    ax = axes[i]
    df.boxplot(column=col, by='is_premium', ax=ax, patch_artist=True,
               boxprops=dict(facecolor='lightblue', color='navy'),
               medianprops=dict(color='red', linewidth=2))
    ax.set_title(col, fontsize=11)
    ax.set_xlabel('is_premium')
    ax.set_ylabel('')

# Remove the extra subplot
axes[-1].set_visible(False)
fig.suptitle('Feature Distributions by Wine Class (Boxplots)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# Skewness analysis
print("Feature Skewness:")
print(df[features].skew().sort_values(ascending=False).round(3))

### 2.4 Correlation Matrix
We compute the Pearson correlation matrix to identify which physicochemical features possess the strongest linear relationships with our target variable. We also check for multicollinearity between features.

In [ ]:
corr_matrix = df.corr()

target_corr = corr_matrix['is_premium'].sort_values(ascending=False)
print("Feature correlations with 'is_premium':\n")
print(target_corr)

plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Correlation Matrix of Physicochemical Variables')
plt.show()

# Flag high inter-feature correlations (potential multicollinearity)
print("\nHigh inter-feature correlations (|r| > 0.6):")
for i in range(len(features)):
    for j in range(i+1, len(features)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.6:
            print(f"  {features[i]} <-> {features[j]}: r = {r:.3f}")

## 3. Data Preprocessing

Before feeding the data into our models, we prepare it with three critical steps:

1. **Train/Test Split with Stratification:** We reserve 20% of data for testing. Because our target is highly imbalanced (~13.5% Premium), we use a stratified split to guarantee both sets maintain this proportion.
2. **SMOTE (Synthetic Minority Oversampling):** To address the class imbalance, we apply SMOTE to the training set only. SMOTE generates synthetic samples for the minority class by interpolating between existing minority samples in feature space. This is applied *after* the train/test split to prevent data leakage.
3. **Feature Scaling (Standardization):** Our features have vastly different scales. Distance-based algorithms (KNN, SVM) and neural networks require scaled inputs. We apply `StandardScaler`, fitting only on training data to prevent leakage.

**Important:** SMOTE and scaling are applied only to the training data. The test set remains untouched and reflects the true class distribution.

In [ ]:
# Define the feature matrix (X) and the target vector (y)
X = df.drop('is_premium', axis=1)
y = df['is_premium']

# 1. Train/Test Split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 2. Apply SMOTE to the training set ONLY
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("Data Preprocessing Completed Successfully!\n")
print(f"Original training set: {X_train.shape[0]} samples")
print(f"  Class 0: {sum(y_train == 0)}, Class 1: {sum(y_train == 1)}")
print(f"\nAfter SMOTE resampling: {X_train_resampled.shape[0]} samples")
print(f"  Class 0: {sum(y_train_resampled == 0)}, Class 1: {sum(y_train_resampled == 1)}")

# 3. Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_resampled)
X_test_scaled = scaler.transform(X_test)

# Also keep unscaled versions for tree-based models
X_train_unscaled = X_train_resampled.values if hasattr(X_train_resampled, 'values') else X_train_resampled
X_test_unscaled = X_test.values if hasattr(X_test, 'values') else X_test

print(f"\nTraining set shape (scaled): {X_train_scaled.shape}")
print(f"Testing set shape (scaled): {X_test_scaled.shape}")
print(f"Premium wines in Test set: {sum(y_test)} ({sum(y_test)/len(y_test)*100:.2f}%)")

## 4. Baseline Model

Before building complex models, we establish a baseline using `DummyClassifier`. This gives us a floor performance that any useful model must beat. We test two strategies: "most_frequent" (always predicts the majority class) and "stratified" (predicts randomly according to class proportions). This contextualizes all subsequent results.

In [ ]:
# Baseline: DummyClassifier
for strategy in ['most_frequent', 'stratified']:
    dummy = DummyClassifier(strategy=strategy, random_state=42)
    dummy.fit(X_train_scaled, y_train_resampled)
    y_pred_dummy = dummy.predict(X_test_scaled)
    
    acc = accuracy_score(y_test, y_pred_dummy)
    f1 = f1_score(y_test, y_pred_dummy, zero_division=0)
    prec = precision_score(y_test, y_pred_dummy, zero_division=0)
    rec = recall_score(y_test, y_pred_dummy, zero_division=0)
    
    print(f"Baseline ({strategy}):")
    print(f"  Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f}\n")

print("Any useful model must substantially beat these baselines.")

## 5. Modeling and Hyperparameter Tuning

In this section, we train and evaluate various machine learning algorithms. To ensure robustness and prevent overfitting, we use `GridSearchCV` with 5-fold stratified cross-validation to systematically search for optimal hyperparameters.

**Crucial Note on Evaluation Metric:** Because our dataset is heavily imbalanced (only ~13.5% Premium wines in the test set), standard accuracy is misleading. A naive model predicting "Standard" 100% of the time would achieve ~86.5% accuracy but completely fail our business objective. Therefore, we instruct `GridSearchCV` to optimize based on the **F1-Score** of the positive class (Premium), which balances precision and recall.

**Note on Class Imbalance Handling:** In addition to SMOTE applied during preprocessing, we also use `class_weight='balanced'` where supported. This provides the model with an additional signal about the importance of the minority class. For tree-based models that don't need scaled data, we use the unscaled (but SMOTE-resampled) training set.

### 5.1 K-Nearest Neighbors (KNN)
KNN classifies a new wine based on the majority class of its $k$ closest neighbors in the feature space. We test different values for the number of neighbors ($k$), distance weightings, and distance metrics.

In [ ]:
# 1. Define the model
knn = KNeighborsClassifier()

# 2. Define the hyperparameter grid
knn_param_grid = {
    'n_neighbors': np.arange(3, 22, 2),
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

# 3. GridSearchCV targeting the F1-score (binary, positive class = 1)
print("Running GridSearchCV for KNN...")
knn_grid = GridSearchCV(
    estimator=knn,
    param_grid=knn_param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

# 4. Fit on SCALED, SMOTE-resampled training data
knn_grid.fit(X_train_scaled, y_train_resampled)

# 5. Extract best parameters and predict
best_knn = knn_grid.best_estimator_
print(f"Best KNN Parameters: {knn_grid.best_params_}")
print(f"Best CV F1-Score: {knn_grid.best_score_:.4f}")

y_pred_knn = best_knn.predict(X_test_scaled)

# 6. Evaluate
print("\n--- KNN Classification Report (Test Set) ---")
print(classification_report(y_test, y_pred_knn))

# 7. Confusion Matrix
plt.figure(figsize=(6, 4))
sns.heatmap(confusion_matrix(y_test, y_pred_knn), annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('KNN Confusion Matrix')
plt.xlabel('Predicted Label (0 = Standard, 1 = Premium)')
plt.ylabel('True Label (0 = Standard, 1 = Premium)')
plt.show()

### 5.1.1 KNN - Technical Insights

**Technical Observations:**
* **Scaling Impact:** As a distance-based algorithm, the `StandardScaler` step was crucial. Without it, features like *Total Sulfur Dioxide* would have overshadowed smaller-scale features like *Chlorides*.
* **SMOTE Effect:** With the original imbalanced data, KNN tended to have very few minority neighbors in most regions of the feature space. SMOTE generates synthetic minority samples that populate these neighborhoods, improving recall significantly.
* **The Dimensionality Challenge:** KNN can struggle with many features due to the "curse of dimensionality," where distance becomes less discriminative. However, 11 features is moderate—the larger challenge here was the class overlap in feature space, where standard and premium wines share similar chemical profiles.

**Result:** KNN serves as a useful baseline beyond the DummyClassifier, but its reliance on local geometry limits its ability to capture the complex chemical synergy that defines a Premium wine.

### 5.2 Support Vector Machines (SVM)

Support Vector Machines find the optimal hyperplane that separates our two classes in the feature space. Since chemical interactions determining taste are rarely linear, we test both a `linear` kernel and an `rbf` (Radial Basis Function) kernel. We use `class_weight='balanced'` to give additional emphasis to the minority class. Key hyperparameters:
* **$C$ (Regularization):** Controls the trade-off between margin width and classification errors.
* **$\gamma$ (Gamma):** Defines how far the influence of a single training example reaches (RBF only).

In [ ]:
# 1. Define the SVM model with class_weight='balanced'
svm = SVC(random_state=42, class_weight='balanced')

# 2. Define the hyperparameter grid
svm_param_grid = [
    {'kernel': ['linear'], 'C': [0.1, 1, 10]},
    {'kernel': ['rbf'], 'C': [0.1, 1, 10, 100], 'gamma': ['scale', 'auto', 0.01, 0.1]}
]

# 3. GridSearchCV
print("Running GridSearchCV for SVM...")
svm_grid = GridSearchCV(
    estimator=svm,
    param_grid=svm_param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

# 4. Fit on SCALED, SMOTE-resampled data
svm_grid.fit(X_train_scaled, y_train_resampled)

best_svm = svm_grid.best_estimator_
print(f"Best SVM Parameters: {svm_grid.best_params_}")
print(f"Best CV F1-Score: {svm_grid.best_score_:.4f}")

# Show linear vs RBF comparison
cv_results = pd.DataFrame(svm_grid.cv_results_)
for kernel in ['linear', 'rbf']:
    mask = cv_results['param_kernel'] == kernel
    best_for_kernel = cv_results.loc[mask, 'mean_test_score'].max()
    print(f"  Best CV F1 with {kernel} kernel: {best_for_kernel:.4f}")

y_pred_svm = best_svm.predict(X_test_scaled)

print("\n--- SVM Classification Report (Test Set) ---")
print(classification_report(y_test, y_pred_svm))

plt.figure(figsize=(6, 4))
sns.heatmap(confusion_matrix(y_test, y_pred_svm), annot=True, fmt='d', cmap='Oranges', cbar=False)
plt.title('SVM Confusion Matrix')
plt.xlabel('Predicted Label (0 = Standard, 1 = Premium)')
plt.ylabel('True Label (0 = Standard, 1 = Premium)')
plt.show()

### 5.2.1 SVM - Technical Insights & Performance Analysis

**Key Findings from Tuning:**
* **Kernel Effectiveness:** The comparison above quantifies the gap between the linear and RBF kernel. The RBF kernel's superiority confirms that the boundary between "Standard" and "Premium" wines is not a simple hyperplane but a complex, non-linear surface in feature space.
* **`class_weight='balanced'`:** Combined with SMOTE, this provides a two-pronged approach to imbalance. SMOTE adds synthetic samples; balanced class weights penalize misclassifications of the minority class more heavily during optimization. This combination typically improves recall without catastrophically hurting precision.
* **The Role of C & Gamma:** Higher C values allow the model to fit tighter boundaries (risking overfitting), while gamma controls the "reach" of each support vector. The grid search found the optimal balance for our data.

### 5.3 Decision Tree Classifier

Decision Trees make sequential, hierarchical splits based on individual features, mimicking human logical reasoning. A major advantage is that they are scale-invariant—they do not require standardized data. We train on the SMOTE-resampled but **unscaled** training data, preserving original units for interpretable rule extraction (e.g., "If alcohol > 11% and pH < 3.3, then Premium").

We use `class_weight='balanced'` and constrain `max_depth` and `min_samples_leaf` via GridSearchCV to prevent overfitting.

In [ ]:
# 1. Decision Tree with class_weight='balanced'
dt = DecisionTreeClassifier(random_state=42, class_weight='balanced')

# 2. Hyperparameter grid
dt_param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': np.arange(3, 10),
    'min_samples_leaf': [5, 10, 20, 30]
}

# 3. GridSearchCV
print("Running GridSearchCV for Decision Tree...")
dt_grid = GridSearchCV(
    estimator=dt,
    param_grid=dt_param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

# 4. Fit on UNSCALED, SMOTE-resampled data
dt_grid.fit(X_train_unscaled, y_train_resampled)

best_dt = dt_grid.best_estimator_
print(f"Best Decision Tree Parameters: {dt_grid.best_params_}")
print(f"Best CV F1-Score: {dt_grid.best_score_:.4f}")

# Predict on UNSCALED test data
y_pred_dt = best_dt.predict(X_test_unscaled)

print("\n--- Decision Tree Classification Report (Test Set) ---")
print(classification_report(y_test, y_pred_dt))

plt.figure(figsize=(6, 4))
sns.heatmap(confusion_matrix(y_test, y_pred_dt), annot=True, fmt='d', cmap='Greens', cbar=False)
plt.title('Decision Tree Confusion Matrix')
plt.xlabel('Predicted Label (0 = Standard, 1 = Premium)')
plt.ylabel('True Label (0 = Standard, 1 = Premium)')
plt.show()

# Visualize the tree
print("\nDecision Tree Structure (top splits reveal key chemical thresholds):")
plt.figure(figsize=(20, 10))
plot_tree(
    best_dt,
    feature_names=X.columns,
    class_names=['Standard', 'Premium'],
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title("Optimized Decision Tree Structure")
plt.show()

# Extract top split thresholds
tree = best_dt.tree_
print("\nTop-3 most important split features:")
importances = best_dt.feature_importances_
top_features = np.argsort(importances)[::-1][:3]
for rank, idx in enumerate(top_features, 1):
    print(f"  {rank}. {X.columns[idx]}: importance = {importances[idx]:.4f}")

### 5.3.1 Decision Tree - Technical Insights

**Technical Observations:**
* **Rule Extraction:** By training on original units, the model reveals interpretable chemical thresholds. The top-level splits are shown above with their feature importances—typically driven by **Alcohol** and **Volatile Acidity**.
* **The Overfitting Challenge:** Despite constraining `max_depth`, a single tree is inherently unstable—small data perturbations can produce very different tree structures (high variance). This motivates the ensemble approach in the next section.
* **`class_weight='balanced'`:** For the Decision Tree, balanced weights shift the split criteria to be more sensitive to the minority class, producing trees that are better at detecting premium wines at the cost of some precision.

**Result:** The Decision Tree is a valuable diagnostic tool providing human-readable logic, but its high variance makes it less reliable as a standalone predictor.

### 5.4 Random Forest Classifier (Ensemble Method)

A Random Forest builds a "forest" of multiple decision trees, where each tree is trained on a random subset of the data (Bagging) and uses a random subset of features at each split. The final prediction is made by majority vote. This **reduces variance** through the averaging effect: while individual trees may overfit, their errors are decorrelated (due to feature subsampling) and cancel out on average.

Like the Decision Tree, Random Forest does not require scaled data. We also extract **Feature Importance** rankings.

In [ ]:
# 1. Random Forest with class_weight='balanced'
rf = RandomForestClassifier(random_state=42, class_weight='balanced')

# 2. Hyperparameter grid (finer granularity than before)
rf_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 5, 10]
}

# 3. GridSearchCV
print("Running GridSearchCV for Random Forest (this may take a moment)...")
rf_grid = GridSearchCV(
    estimator=rf,
    param_grid=rf_param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

# 4. Fit on UNSCALED, SMOTE-resampled data
rf_grid.fit(X_train_unscaled, y_train_resampled)

best_rf = rf_grid.best_estimator_
print(f"Best Random Forest Parameters: {rf_grid.best_params_}")
print(f"Best CV F1-Score: {rf_grid.best_score_:.4f}")

y_pred_rf = best_rf.predict(X_test_unscaled)

print("\n--- Random Forest Classification Report (Test Set) ---")
print(classification_report(y_test, y_pred_rf))

plt.figure(figsize=(6, 4))
sns.heatmap(confusion_matrix(y_test, y_pred_rf), annot=True, fmt='d', cmap='Purples', cbar=False)
plt.title('Random Forest Confusion Matrix')
plt.xlabel('Predicted Label (0 = Standard, 1 = Premium)')
plt.ylabel('True Label (0 = Standard, 1 = Premium)')
plt.show()

# Feature Importance
print("\nFeature Importance Ranking:")
importances = best_rf.feature_importances_
feature_imp_df = pd.DataFrame({'Feature': X.columns, 'Importance': importances})
feature_imp_df = feature_imp_df.sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_imp_df, palette='magma')
plt.title('Random Forest Feature Importance: What Makes a Premium Wine?')
plt.xlabel('Relative Importance')
plt.ylabel('Physicochemical Feature')
plt.show()

### 5.4.1 Random Forest - Technical Insights

**Why Random Forest Outperforms a Single Tree (Mathematical Intuition):**

The variance of an average of $B$ trees, each with variance $\sigma^2$ and pairwise correlation $\rho$, is:

$$\text{Var}(\bar{f}) = \rho \sigma^2 + \frac{1-\rho}{B} \sigma^2$$

Random Forest reduces variance through two mechanisms: (1) increasing $B$ (more trees) shrinks the second term; (2) random feature subsampling at each split reduces $\rho$ (correlation between trees), shrinking the first term. This is why Random Forest consistently outperforms a single Decision Tree.

**Technical Observations:**
* **From Instability to Stability:** By aggregating many trees, the model eliminates the high variance of individual trees. The "majority vote" cancels out individual overfitting errors.
* **Feature Importance Insights:** The forest identifies the top predictors, providing actionable targets for quality control.
* **`class_weight='balanced'` + SMOTE:** The combined approach ensures that both the data distribution and the loss function emphasize the minority class.

### 5.5 Neural Network (Multi-Layer Perceptron)

This section introduces a Neural Network built with TensorFlow/Keras. The network uses a "funnel" topology with Dropout regularization and **EarlyStopping** to prevent overfitting.

#### Architectural Design

* **Input Layer (Implicit):** Receives the 11 scaled chemical variables. Scaling is mathematically mandatory to ensure gradient descent converges smoothly.
* **Hidden Layers (16 → 8 Neurons):** The first layer learns basic chemical interactions; the second compresses these into abstract features. The low neuron count restricts capacity to prevent memorization.
* **Dropout Layer (0.2):** Randomly drops 20% of neurons during training, forcing robust, distributed representations rather than over-reliance on any single feature.
* **Output Layer (1 Neuron, Sigmoid):** Outputs a probability in [0, 1] for the Premium class.

#### Learning Engine
* **Loss Function (Binary Crossentropy):** Applies logarithmic penalty for incorrect predictions.
* **Adam Optimizer:** Adapts learning rates per-parameter for fast, stable convergence.
* **EarlyStopping:** Monitors validation loss and stops training when it stops improving, preventing the model from memorizing training noise.

In [ ]:
# ==============================================================================
# NEURAL NETWORK IMPLEMENTATION & TRAINING
# ==============================================================================

# 1. Model Architecture
nn_model = Sequential(name="Wine_Quality_Predictor")
nn_model.add(Dense(units=16, input_dim=X_train_scaled.shape[1], activation='relu', name='Hidden_Layer_1'))
nn_model.add(Dropout(rate=0.2, name='Dropout_Regularization'))
nn_model.add(Dense(units=8, activation='relu', name='Hidden_Layer_2'))
nn_model.add(Dense(units=1, activation='sigmoid', name='Output_Layer'))

# 2. Compilation
custom_optimizer = Adam(learning_rate=0.005)
nn_model.compile(optimizer=custom_optimizer, loss='binary_crossentropy', metrics=['accuracy'])

print("--- Neural Network Architecture ---")
nn_model.summary()

# 3. EarlyStopping callback
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=7,
    restore_best_weights=True,
    verbose=1
)

# 4. Training with stratified validation
# We manually create a stratified validation split for Keras
from sklearn.model_selection import StratifiedShuffleSplit
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_idx, val_idx = next(sss.split(X_train_scaled, y_train_resampled))

X_nn_train, X_nn_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
y_nn_train, y_nn_val = y_train_resampled.values[train_idx], y_train_resampled.values[val_idx]

print(f"\nNN Training set: {len(X_nn_train)} samples (Class 1: {sum(y_nn_train)})")
print(f"NN Validation set: {len(X_nn_val)} samples (Class 1: {sum(y_nn_val)})")

print("\nInitiating Neural Network Training...")
history = nn_model.fit(
    X_nn_train, y_nn_train,
    epochs=100,  # More epochs since EarlyStopping will cut it
    batch_size=32,
    validation_data=(X_nn_val, y_nn_val),
    callbacks=[early_stop],
    verbose=0
)
print(f"Training stopped at epoch {len(history.history['loss'])}")

# 5. Evaluation
y_pred_prob_nn = nn_model.predict(X_test_scaled).ravel()
y_pred_nn = (y_pred_prob_nn > 0.5).astype(int)

print("\n--- Neural Network Classification Report (Test Set) ---")
print(classification_report(y_test, y_pred_nn))

# 6. Confusion Matrix
plt.figure(figsize=(6, 4))
sns.heatmap(confusion_matrix(y_test, y_pred_nn), annot=True, fmt='d', cmap='Blues',
            xticklabels=['Standard (0)', 'Premium (1)'],
            yticklabels=['Standard (0)', 'Premium (1)'])
plt.title('Neural Network Confusion Matrix', fontsize=14)
plt.ylabel('Actual Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.show()

# 7. Learning Curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'], label='Training Loss', color='darkorange', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validation Loss', color='navy', linewidth=2)
axes[0].set_title('Loss over Epochs')
axes[0].set_xlabel('Epochs')
axes[0].set_ylabel('Binary Crossentropy Loss')
axes[0].legend()
axes[0].grid(True, linestyle='--', alpha=0.7)

axes[1].plot(history.history['accuracy'], label='Training Accuracy', color='darkorange', linewidth=2)
axes[1].plot(history.history['val_accuracy'], label='Validation Accuracy', color='navy', linewidth=2)
axes[1].set_title('Accuracy over Epochs')
axes[1].set_xlabel('Epochs')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.7)

plt.suptitle('Neural Network Diagnostics: Learning Curves', fontsize=14)
plt.tight_layout()
plt.show()

### 5.5.1 Neural Network - Technical Insights

**Technical Observations:**
* **EarlyStopping:** The model trained until validation loss plateaued, then restored the best weights. This is more principled than running a fixed number of epochs, as it adapts to the specific convergence dynamics of each run.
* **Stratified Validation:** Unlike the default `validation_split` in Keras (which takes the last N% of samples without stratification), we explicitly created a stratified validation split. With only ~13.5% positive samples, a non-stratified split could produce a validation set with very few or no premium wines, leading to unreliable validation metrics.
* **Dropout Effectiveness:** The learning curves should show training and validation loss tracking closely—evidence that the 20% Dropout rate successfully prevented overfitting.

**Result:** The Neural Network typically achieves strong recall (good at finding premium wines) but may sacrifice some precision. It serves as the "scout" model that ensures no premium wine is missed, even at the cost of occasional false alarms.

## 6. Comprehensive Model Comparison

This section aggregates all model metrics and adds the DummyClassifier baselines for context. We compare Accuracy, Precision, Recall, and F1-Score.

In [ ]:
# ==============================================================================
# AGGREGATE METRICS & BENCHMARKING
# ==============================================================================

def generate_metrics(y_true, y_pred):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1-Score': f1_score(y_true, y_pred, zero_division=0)
    }

# Add baseline
dummy_mf = DummyClassifier(strategy='most_frequent', random_state=42)
dummy_mf.fit(X_train_scaled, y_train_resampled)
y_pred_baseline = dummy_mf.predict(X_test_scaled)

benchmark_results = {
    'Baseline (majority)': generate_metrics(y_test, y_pred_baseline),
    'KNN': generate_metrics(y_test, y_pred_knn),
    'SVM (RBF)': generate_metrics(y_test, y_pred_svm),
    'Decision Tree': generate_metrics(y_test, y_pred_dt),
    'Random Forest': generate_metrics(y_test, y_pred_rf),
    'Neural Network': generate_metrics(y_test, y_pred_nn)
}

benchmark_df = pd.DataFrame(benchmark_results).T
benchmark_df = benchmark_df.round(4)

print("--- Model Performance Benchmark ---")
print("(Highlighted: best score per metric, excluding baseline)\n")
display(benchmark_df.style.highlight_max(color='lightcoral', axis=0, subset=benchmark_df.index[1:]))

### 6.1 Interpreting the Benchmark Results

Key observations from the benchmark table:

1. **Baseline Context:** The majority-class baseline achieves ~86.5% accuracy but 0.0 F1-Score on the minority class. This proves that raw accuracy is meaningless for imbalanced problems and justifies our focus on F1-Score.
2. **The Ensemble Advantage (Random Forest):** Random Forest typically dominates F1-Score by leveraging bootstrap aggregating and feature subsampling to capture complex non-linear thresholds while neutralizing variance.
3. **The Margin & Manifold Seekers (SVM & NN):** Both project data into higher-dimensional spaces. Their performance depends on proper standardization, making them slightly more brittle in production than tree-based methods.
4. **The Baseline Models (DT & KNN):** The single Decision Tree provides interpretability but suffers from variance. KNN, as a lazy learner, is sensitive to feature space geometry and class overlap.

**Important caveat:** With only ~43 positive samples in the test set, differences of a few percentage points in F1 could be due to random variation. Ideally, nested cross-validation or bootstrap confidence intervals would strengthen these comparisons.

## 7. ROC and Precision-Recall Curves

We plot both **ROC curves** and **Precision-Recall (PR) curves** for all models.

**Why both?** ROC-AUC can be misleadingly optimistic for imbalanced datasets because it includes the True Negative Rate, which is inflated by the large majority class. Precision-Recall curves focus exclusively on the positive class and provide a more honest picture of model performance when the minority class is what we care about.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# ==================== ROC CURVES ====================
ax1 = axes[0]

# KNN
y_prob_knn = best_knn.predict_proba(X_test_scaled)[:, 1]
fpr_knn, tpr_knn, _ = roc_curve(y_test, y_prob_knn)
auc_knn = auc(fpr_knn, tpr_knn)
ax1.plot(fpr_knn, tpr_knn, label=f'KNN (AUC = {auc_knn:.3f})', color='blue')

# SVM (decision_function since probability=False by default)
y_score_svm = best_svm.decision_function(X_test_scaled)
fpr_svm, tpr_svm, _ = roc_curve(y_test, y_score_svm)
auc_svm = auc(fpr_svm, tpr_svm)
ax1.plot(fpr_svm, tpr_svm, label=f'SVM (AUC = {auc_svm:.3f})', color='orange')

# Decision Tree
y_prob_dt = best_dt.predict_proba(X_test_unscaled)[:, 1]
fpr_dt, tpr_dt, _ = roc_curve(y_test, y_prob_dt)
auc_dt = auc(fpr_dt, tpr_dt)
ax1.plot(fpr_dt, tpr_dt, label=f'Decision Tree (AUC = {auc_dt:.3f})', color='green')

# Random Forest
y_prob_rf = best_rf.predict_proba(X_test_unscaled)[:, 1]
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)
auc_rf = auc(fpr_rf, tpr_rf)
ax1.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {auc_rf:.3f})', color='purple', linewidth=2.5)

# Neural Network
fpr_nn, tpr_nn, _ = roc_curve(y_test, y_pred_prob_nn)
auc_nn = auc(fpr_nn, tpr_nn)
ax1.plot(fpr_nn, tpr_nn, label=f'Neural Network (AUC = {auc_nn:.3f})', color='teal', linewidth=2)

ax1.plot([0, 1], [0, 1], 'k--', label='Random Guess (AUC = 0.500)')
ax1.set_title('ROC Curve Comparison', fontsize=14)
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate (Recall)')
ax1.legend(loc='lower right', fontsize=10)
ax1.grid(alpha=0.3)

# ==================== PRECISION-RECALL CURVES ====================
ax2 = axes[1]

# Baseline: proportion of positives
baseline_pr = y_test.mean()

prec_knn, rec_knn, _ = precision_recall_curve(y_test, y_prob_knn)
ap_knn = average_precision_score(y_test, y_prob_knn)
ax2.plot(rec_knn, prec_knn, label=f'KNN (AP = {ap_knn:.3f})', color='blue')

prec_svm, rec_svm, _ = precision_recall_curve(y_test, y_score_svm)
ap_svm = average_precision_score(y_test, y_score_svm)
ax2.plot(rec_svm, prec_svm, label=f'SVM (AP = {ap_svm:.3f})', color='orange')

prec_dt, rec_dt, _ = precision_recall_curve(y_test, y_prob_dt)
ap_dt = average_precision_score(y_test, y_prob_dt)
ax2.plot(rec_dt, prec_dt, label=f'Decision Tree (AP = {ap_dt:.3f})', color='green')

prec_rf, rec_rf, _ = precision_recall_curve(y_test, y_prob_rf)
ap_rf = average_precision_score(y_test, y_prob_rf)
ax2.plot(rec_rf, prec_rf, label=f'Random Forest (AP = {ap_rf:.3f})', color='purple', linewidth=2.5)

prec_nn, rec_nn, _ = precision_recall_curve(y_test, y_pred_prob_nn)
ap_nn = average_precision_score(y_test, y_pred_prob_nn)
ax2.plot(rec_nn, prec_nn, label=f'Neural Network (AP = {ap_nn:.3f})', color='teal', linewidth=2)

ax2.axhline(y=baseline_pr, color='k', linestyle='--', label=f'Baseline (AP = {baseline_pr:.3f})')
ax2.set_title('Precision-Recall Curve Comparison', fontsize=14)
ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.legend(loc='upper right', fontsize=10)
ax2.grid(alpha=0.3)

plt.suptitle('Model Discrimination: ROC vs Precision-Recall', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print("Note: For imbalanced problems, the Precision-Recall curve (right) is more")
print("informative than ROC (left), as it focuses exclusively on the minority class.")

## 8. Threshold Optimization

By default, classifiers use a 0.5 probability threshold. However, the optimal threshold depends on the **business cost** of false positives vs. false negatives:
* **False Positive (FP):** A standard wine labeled as premium → brand reputation damage, customer disappointment.
* **False Negative (FN):** A premium wine labeled as standard → revenue loss from underpricing.

We plot F1-Score as a function of the decision threshold for the best model and identify the optimal operating point.

In [ ]:
# ==============================================================================
# THRESHOLD OPTIMIZATION (for Random Forest as our best model)
# ==============================================================================

# Get probability scores from the best model (Random Forest)
y_prob_best = best_rf.predict_proba(X_test_unscaled)[:, 1]

thresholds = np.arange(0.05, 0.96, 0.01)
f1_scores = []
precision_scores_list = []
recall_scores_list = []

for t in thresholds:
    y_pred_t = (y_prob_best >= t).astype(int)
    f1_scores.append(f1_score(y_test, y_pred_t, zero_division=0))
    precision_scores_list.append(precision_score(y_test, y_pred_t, zero_division=0))
    recall_scores_list.append(recall_score(y_test, y_pred_t, zero_division=0))

# Find optimal threshold
optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[optimal_idx]
optimal_f1 = f1_scores[optimal_idx]

plt.figure(figsize=(12, 6))
plt.plot(thresholds, f1_scores, label='F1-Score', color='purple', linewidth=2.5)
plt.plot(thresholds, precision_scores_list, label='Precision', color='blue', linewidth=1.5, alpha=0.7)
plt.plot(thresholds, recall_scores_list, label='Recall', color='red', linewidth=1.5, alpha=0.7)
plt.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5, label='Default threshold (0.5)')
plt.axvline(x=optimal_threshold, color='green', linestyle='-', linewidth=2, label=f'Optimal threshold ({optimal_threshold:.2f})')
plt.scatter([optimal_threshold], [optimal_f1], color='green', s=100, zorder=5)
plt.title('Threshold Optimization: F1-Score vs Decision Threshold (Random Forest)', fontsize=14)
plt.xlabel('Decision Threshold', fontsize=12)
plt.ylabel('Score', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

print(f"Optimal Threshold: {optimal_threshold:.2f}")
print(f"F1-Score at optimal threshold: {optimal_f1:.4f}")
print(f"F1-Score at default 0.5 threshold: {f1_score(y_test, y_pred_rf):.4f}")

# Re-evaluate with optimal threshold
y_pred_optimal = (y_prob_best >= optimal_threshold).astype(int)
print(f"\n--- Random Forest at Optimal Threshold ({optimal_threshold:.2f}) ---")
print(classification_report(y_test, y_pred_optimal))

## 9. Hyperparameter Sensitivity Analysis

This section explores how varying key hyperparameters impacts performance and stability. Validation curves visualize the bias-variance tradeoff: when training score is high but validation score is low, the model is overfitting.

**Consistency note:** We use scaled data for distance-based models (KNN, SVM) and **unscaled** data for tree-based models (Decision Tree, Random Forest), consistent with how each model was actually trained.

In [ ]:
# ==============================================================================
# VALIDATION CURVES: OVERFITTING VS UNDERFITTING DIAGNOSTICS
# ==============================================================================

def plot_hyperparameter_curve(estimator, title, param_name, param_range, X, y, cv=5):
    train_scores, test_scores = validation_curve(
        estimator, X, y, param_name=param_name, param_range=param_range,
        cv=cv, scoring="f1", n_jobs=-1
    )
    train_mean = np.mean(train_scores, axis=1)
    train_std = np.std(train_scores, axis=1)
    test_mean = np.mean(test_scores, axis=1)
    test_std = np.std(test_scores, axis=1)

    plt.plot(param_range, train_mean, label="Training F1-Score", color="darkred", marker='o', linewidth=2)
    plt.fill_between(param_range, train_mean - train_std, train_mean + train_std, color="darkred", alpha=0.1)
    plt.plot(param_range, test_mean, label="CV F1-Score", color="steelblue", marker='o', linewidth=2)
    plt.fill_between(param_range, test_mean - test_std, test_mean + test_std, color="steelblue", alpha=0.1)
    plt.title(title, fontsize=13)
    plt.xlabel(param_name, fontsize=11)
    plt.ylabel("F1-Score", fontsize=11)
    plt.legend(loc="best")
    plt.grid(True, linestyle='--', alpha=0.6)

plt.figure(figsize=(18, 12))

# Random Forest - UNSCALED data (consistent with training)
plt.subplot(2, 2, 1)
plot_hyperparameter_curve(
    RandomForestClassifier(random_state=42, class_weight='balanced'),
    "Random Forest: n_estimators vs. Performance",
    "n_estimators", [10, 50, 100, 200, 300],
    X_train_unscaled, y_train_resampled
)

# KNN - SCALED data (consistent with training)
plt.subplot(2, 2, 2)
plot_hyperparameter_curve(
    KNeighborsClassifier(),
    "KNN: n_neighbors vs. Performance",
    "n_neighbors", [1, 5, 10, 20, 50],
    X_train_scaled, y_train_resampled
)

# Decision Tree - UNSCALED data (consistent with training)
plt.subplot(2, 2, 3)
plot_hyperparameter_curve(
    DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    "Decision Tree: max_depth vs. Performance",
    "max_depth", np.arange(1, 21),
    X_train_unscaled, y_train_resampled
)

# SVM - SCALED data (consistent with training)
plt.subplot(2, 2, 4)
plot_hyperparameter_curve(
    SVC(kernel='rbf', random_state=42, class_weight='balanced'),
    "SVM (RBF): C vs. Performance",
    "C", [0.01, 0.1, 1, 10, 100],
    X_train_scaled, y_train_resampled
)

plt.tight_layout()
plt.show()

**A Note on Neural Network Diagnostics:** The Neural Network is intentionally absent from these 1D validation curves. In deep learning, the standard diagnostic is the **learning curve over epochs** (training vs validation loss), which we already generated in Section 5.5. Our learning curves confirmed that the network converged smoothly and that Dropout + EarlyStopping successfully prevented overfitting. Running scikit-learn's `validation_curve` on a Keras model would be computationally expensive and redundant.

## 10. Final Conclusion & Business Recommendations

### 10.1 Model Performance Summary

| Model | Best Attribute | Business Value |
| :--- | :--- | :--- |
| **Random Forest** | **Best F1-Score** | The most balanced and reliable "All-Rounder". |
| **Neural Network** | **High Recall** | The best "Scout" for finding every potential premium bottle. |
| **SVM (RBF)** | **High Precision** | The most "Conservative" judge, rarely making false claims. |
| **Decision Tree** | **Interpretability** | Provides clear chemical "Rules of Thumb" for winemakers. |
| **KNN** | **Simplicity** | A useful geometric baseline. |
| **Baseline** | **Reference** | Proves all models add substantial value over naive prediction. |

### 10.2 The Winner: Random Forest
For deployment, we recommend the **Random Forest** model with the optimized decision threshold identified in Section 8. The ensemble approach provides the necessary stability to handle the complex, non-linear chemistry of the dataset. Combined with SMOTE and balanced class weights, it effectively handles the class imbalance while identifying the critical synergy between **Alcohol**, **Sulphates**, and **Volatile Acidity**.

### 10.3 Strategic Insights for the Winery
1. **The Alcohol-Quality Correlation:** Higher quality is strongly associated with specific alcohol thresholds. Monitoring fermentation to reach these levels is crucial.
2. **Sulphates as a Quality Driver:** Precise control over additive levels is a hallmark of "Premium" wines.
3. **Acidity Management:** Volatile Acidity (acetic acid) acts as a strong negative indicator. Keeping this value low is the most statistically significant way to avoid a "Standard" classification.

### 10.4 Limitations & Future Directions
**Limitations of this analysis:**
* The test set contains only ~43 premium wine samples, making metric differences between models potentially unstable. Nested cross-validation would provide more robust model comparisons.
* Predicted probabilities may not be well-calibrated—a separate calibration analysis (e.g., Platt scaling or isotonic regression) would be needed before using probabilities in production.
* A single 80/20 train/test split introduces variance in the results.

**Future improvements:**
* **Feature Engineering:** Creating interaction terms (e.g., alcohol × volatile acidity) or a composite "Balance Index" combining pH and acidity.
* **Data Collection:** Acquiring more Premium wine samples to reduce reliance on synthetic oversampling.
* **Gradient Boosting:** Testing XGBoost or LightGBM, which often outperform Random Forests on tabular data.
* **Deployment:** Integrating the optimized Random Forest into the winery's quality control pipeline for real-time grading.